Create a TS+biology tracers initial file for one bathymetry based on a restart file from a different bathymetry

In [26]:
import netCDF4 as nc
import xarray as xr
import numpy as np 

from salishsea_tools import nc_tools

%matplotlib inline

# New Bathymetry (via its mesh mask)

In [27]:
mesh = nc.Dataset('/ocean/atall/MOAD/grid/mesh_mask202605.nc')
mbathy = mesh.variables['mbathy'][0,:,:]
#used to calculate number of vertical ocean grid cells at each (i,j) (1=land point)
gdepw = mesh.variables['gdepw_0'][0,:,:,:]
surface_tmask = mesh.variables['tmask'][0,0,:,:]
surface_tmask = np.abs(surface_tmask-1)
tmask = mesh.variables['tmask'][0,:,:,:]
tmask = np.abs(tmask-1)
lats = mesh.variables['nav_lat'][:]
lons = mesh.variables['nav_lon'][:]
mesh.close()

In [28]:
# calculate bathymetry based on meshmask
NEMO_bathy = np.zeros(mbathy.shape)
for i in range(NEMO_bathy.shape[1]):
    for j in range(NEMO_bathy.shape[0]):
        level = mbathy[j,i]
        NEMO_bathy[j,i] = gdepw[level,j,i]
NEMO_bathy = np.ma.masked_array(NEMO_bathy, mask = surface_tmask)

# Old Bathymetry (based on its mesh mask)

In [29]:
oldmesh = nc.Dataset('/ocean/atall/MOAD/grid/mesh_mask_202310b.nc')
oldmbathy =oldmesh.variables['mbathy'][0,:,:] 
#used to calculate number of vertical ocean grid cells at each (i,j) (1=land point)
oldgdepw = oldmesh.variables['gdepw_0'][0,:,:,:]
oldsurface_tmask = oldmesh.variables['tmask'][0,0,:,:]
oldsurface_tmask = np.abs(oldsurface_tmask-1)
oldtmask = oldmesh.variables['tmask'][0,:,:,:]
oldtmask = np.abs(oldtmask-1)
oldmesh.close()

In [30]:
np.max(oldmbathy)

np.int16(39)

# Restart Files to Get Tracers (TS and Biology)

In [31]:
dataphys = nc.Dataset('/ocean/atall/MOAD/Model/runs/restart/31dec09/SalishSea_06324480_restart.nc')
databio = nc.Dataset('/ocean/atall/MOAD/Model/runs/restart/31dec09/SalishSea_06324480_restart_trc.nc')   

In [32]:
physical = ['tn', 'sn',
           'tb', 'sb']
biological = ['TRNDON', 'TRNMICZ','TRNMYRI','TRNNH4','TRNNO3','TRNTRA',
              'TRNPHY','TRNDIAT','TRNPON','TRNSi','TRNbSi',
              'TRNDIC', 'TRNTA', 'TRNO2',
             'TRBDON', 'TRBMICZ','TRBMYRI','TRBNH4','TRBNO3','TRBTRA',
              'TRBPHY','TRBDIAT','TRBPON','TRBSi','TRBbSi',
             'TRBDIC', 'TRBTA', 'TRBO2']
varas = {}
for vb in physical:
    varas[vb] = dataphys.variables[vb][0, :]
for vb in biological:
    print (vb)
    varas[vb] = databio.variables[vb][0, :]
#dataphys.close()
#databio.close()
varall = physical + biological

TRNDON
TRNMICZ
TRNMYRI
TRNNH4
TRNNO3
TRNTRA
TRNPHY
TRNDIAT
TRNPON
TRNSi
TRNbSi
TRNDIC
TRNTA
TRNO2
TRBDON
TRBMICZ
TRBMYRI
TRBNH4
TRBNO3
TRBTRA
TRBPHY
TRBDIAT
TRBPON
TRBSi
TRBbSi
TRBDIC
TRBTA
TRBO2


# Fill in any Missing Data Points

In [33]:
def find_mean(varas, varall, i, j, k, dd, oldtmask):
    for vb in varall:
        imin = max(i-dd, 0)
        imax = min(i+dd, 897)
        jmin = max(j-dd, 0)
        jmax = min(j+dd, 397)
        temporary = np.sum(varas[vb][k, imin:imax+1, jmin:jmax+1]*(1-oldtmask[k, imin:imax+1, jmin:jmax+1]))
        count = np.sum(1-oldtmask[k, imin:imax+1, jmin:jmax+1])
        if count == 0:
            varas[vb][k, i, j] = 0
        else:
            varas[vb][k, i, j] = temporary/count
    return varas

In [34]:
def fillit(kmax, oldtmask, varas, varall):
    dd = 1
    bad = 1
    while bad > 0:
        dd += 1
        good = 1
        while good > 0:
            good = 0; bad = 0; already = 0
            for k in range(kmax+1):
                for i in range(1, 898):
                    for j in range(1, 398):
                        if tmask[k,i,j] < oldtmask[k,i,j]:
                            if varas['sn'][k, i, j] > 0:
                                already = already + 1
                            else:
                                varas = find_mean(varas, varall, i, j, k, dd, oldtmask)
                                if varas['sn'][k, i, j] > 0:
                                    good = good + 1
                                else:
                                    bad = bad + 1
                                    if dd > 5:
                                        print (k, i, j)
            print ('dd', dd, 'good', good)
            print ('already', already, 'bad', bad)

This can take a very long time if the bathymetries are very different, aka add a new long river.  If you want you can do it in pieces by starting with the first argument at say 5 and then slowly increasing it.  You do need to go to 39 finally.

In [35]:
fillit(39, oldtmask, varas, varall)

dd 2 good 319
already 0 bad 23
dd 2 good 0
already 319 bad 23
dd 3 good 16
already 319 bad 7
dd 3 good 0
already 335 bad 7
dd 4 good 4
already 335 bad 3
dd 4 good 0
already 339 bad 3
dd 5 good 1
already 339 bad 2
dd 5 good 0
already 340 bad 2
23 70 118
23 70 119
dd 6 good 0
already 340 bad 2
23 70 118
23 70 119
dd 7 good 0
already 340 bad 2
23 70 118
23 70 119
dd 8 good 0
already 340 bad 2
23 70 118
23 70 119
dd 9 good 0
already 340 bad 2
23 70 118
23 70 119
dd 10 good 0
already 340 bad 2
23 70 119
dd 11 good 1
already 340 bad 1
23 70 119
dd 11 good 0
already 341 bad 1
dd 12 good 1
already 341 bad 0
dd 12 good 0
already 342 bad 0


# Write Initial File

In [36]:
# build nc file
new_initialfile = nc.Dataset('/ocean/atall/MOAD/Model/runs/restart/31dec09/restart_phybio_06324480_31dec09_updated.nc', 'w')

new_initialfile.setncattr('title', 'All tracers for Bathymetry 202605 from nowcast-green 31dec09, with N and Si inc.') 
new_initialfile.setncattr('notebook_name', '/ocean/atall/MOAD/analysis-abdoul/notebooks/bathy/Initial_from_Restart_Bathy202605_Spinup31Dec2009') 
new_initialfile.setncattr('nc_filepath', '/ocean/atall/MOAD/Model/runs/restart/31dec09/restart_phybio_06324480_31dec09_updated.nc')
new_initialfile.setncattr('comment', 'All Tracers, physical and biological')
new_initialfile.createDimension('y', 898)
new_initialfile.createDimension('x', 398)
new_initialfile.createDimension('z', 40)
new_initialfile.createDimension('t', None)

"<class 'netCDF4.Dimension'>" (unlimited): name = 't', size = 0

In [37]:
thevara = {}
for vb in varall:
    thevara[vb] = new_initialfile.createVariable(
        vb, 'float64', ('t', 'z', 'y', 'x'), zlib=True,
        least_significant_digit=1e-5, fill_value=-99)
    thevara[vb][0] = varas[vb]
    print (vb, np.max(thevara[vb]))
new_initialfile

tn 11.0625
sn 34.0625
tb 11.0625
sb 34.0625
TRNDON 9.0
TRNMICZ 0.4375
TRNMYRI 0.0
TRNNH4 7.125
TRNNO3 38.4375
TRNTRA 6.8125
TRNPHY 0.375
TRNDIAT 0.4375
TRNPON 6.5
TRNSi 124.0625
TRNbSi 8.875
TRNDIC 2302.9375
TRNTA 2325.1875
TRNO2 366.125
TRBDON 9.0
TRBMICZ 0.4375
TRBMYRI 0.0
TRBNH4 7.125
TRBNO3 38.4375
TRBTRA 6.8125
TRBPHY 0.375
TRBDIAT 0.4375
TRBPON 6.5
TRBSi 123.875
TRBbSi 8.875
TRBDIC 2302.9375
TRBTA 2325.1875
TRBO2 366.125


<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    title: All tracers for Bathymetry 202605 from nowcast-green 31dec09, with N and Si inc.
    notebook_name: /ocean/atall/MOAD/analysis-abdoul/notebooks/bathy/Initial_from_Restart_Bathy202605_Spinup31Dec2009
    nc_filepath: /ocean/atall/MOAD/Model/runs/restart/31dec09/restart_phybio_06324480_31dec09_updated.nc
    comment: All Tracers, physical and biological
    dimensions(sizes): y(898), x(398), z(40), t(1)
    variables(dimensions): float64 tn(t, z, y, x), float64 sn(t, z, y, x), float64 tb(t, z, y, x), float64 sb(t, z, y, x), float64 TRNDON(t, z, y, x), float64 TRNMICZ(t, z, y, x), float64 TRNMYRI(t, z, y, x), float64 TRNNH4(t, z, y, x), float64 TRNNO3(t, z, y, x), float64 TRNTRA(t, z, y, x), float64 TRNPHY(t, z, y, x), float64 TRNDIAT(t, z, y, x), float64 TRNPON(t, z, y, x), float64 TRNSi(t, z, y, x), float64 TRNbSi(t, z, y, x), float64 TRNDIC(t, z, y, x), float64 TRNTA(t, z, y, x), float64 TRNO2(t, z,

In [38]:
new_initialfile.close()

# Copy variables from the old restart files to the new one and correct salinity

In [39]:
var_phy_to_copy = {'nav_lon','nav_lat','nav_lev','time_counter','rnf_b', 'sfx_b', 'rotn', 'vtau_b', 'rdttra1', 'avt', 'nav_lon', 'qns_b', 'vn', 'hdivn', 'time_counter', 'en', 'hdivb', 'sshb', 'rhop', 'ndastp', 'adatrj', 'fse3t_n', 'ssh_ibb', 'avm', 'avmu', 'mxln', 'sshn', 'sbc_sc_b', 'nav_lev', 'emp_b', 'qsr_hc_b', 'sbc_hc_b', 'rnf_sc_b', 'vb2_b', 'ub2_b', 'nav_lat', 'ub', 'rotb', 'fse3t_b', 'un', 'rdt', 'utau_b', 'fraqsr_1lev', 'avmv', 'vb', 'kt', 'rnf_hc_b'}
var_bio_to_copy = {'rnf_pis_DIC_b', 'rnf_pis_MYRI_b', 'sbc_DON_b', 'sbc_bSi_b', 'nav_lon', 'rnf_pis_PON_b', 'rnf_pis_PHY_b', 'time_counter', 'rnf_pis_MICZ_b', 'sbc_TA_b', 'rdttrc1', 'rnf_pis_TRA_b', 'ndastp', 'sbc_Si_b', 'rnf_pis_TA_b', 'adatrj', 'rnf_pis_bSi_b', 'sbc_PHY_b', 'nav_lev', 'sbc_DIC_b', 'sbc_DIAT_b', 'sbc_O2_b', 'sbc_MYRI_b', 'rnf_pis_NO3_b', 'rnf_pis_DON_b', 'sbc_MICZ_b', 'sbc_PON_b', 'sbc_NH4_b', 'rnf_pis_O2_b', 'nav_lat', 'sbc_TRA_b', 'kt', 'rnf_pis_DIAT_b', 'rnf_pis_Si_b', 'rnf_pis_NH4_b', 'sbc_NO3_b'}

In [40]:
dfphy = xr.open_dataset('/ocean/atall/MOAD/Model/runs/restart/31dec09/SalishSea_06324480_restart.nc') 
dfbio = xr.open_dataset('/ocean/atall/MOAD/Model/runs/restart/31dec09/SalishSea_06324480_restart_trc.nc') 
dfnew = xr.open_dataset('/ocean/atall/MOAD/Model/runs/restart/31dec09/restart_phybio_06324480_31dec09_updated.nc')

In [41]:
dfnew[var_phy_to_copy]=dfphy[var_phy_to_copy]
dfnew[var_bio_to_copy]=dfbio[var_bio_to_copy]
dfnew

<xarray.Dataset> Size: 6GB
Dimensions:         (t: 1, z: 40, y: 898, x: 398)
Dimensions without coordinates: t, z, y, x
Data variables: (12/104)
    tn              (t, z, y, x) float64 114MB ...
    sn              (t, z, y, x) float64 114MB ...
    tb              (t, z, y, x) float64 114MB ...
    sb              (t, z, y, x) float64 114MB ...
    TRNDON          (t, z, y, x) float64 114MB ...
    TRNMICZ         (t, z, y, x) float64 114MB ...
    ...              ...
    rnf_pis_O2_b    (t, y, x) float64 3MB ...
    rdttrc1         float64 8B ...
    sbc_PHY_b       (t, y, x) float64 3MB ...
    sbc_PON_b       (t, y, x) float64 3MB ...
    sbc_O2_b        (t, y, x) float64 3MB ...
    rnf_pis_MYRI_b  (t, y, x) float64 3MB ...
Attributes:
    title:          All tracers for Bathymetry 202605 from nowcast-green 31de...
    notebook_name:  /ocean/atall/MOAD/analysis-abdoul/notebooks/bathy/Initial...
    nc_filepath:    /ocean/atall/MOAD/Model/runs/restart/31dec09/restart_phyb...
    comment:        All Tracers, physical and biological

In [42]:
dfnew.to_netcdf('/ocean/atall/MOAD/Model/runs/restart/31dec09/restart_PHYBIO_06324480_31dec09_bathy202605.nc', mode='w')

In [43]:
#
Sn = dfnew.variables['sn']
Sb = dfnew.variables['sb']
Sn.shape

(1, 40, 898, 398)

In [44]:
kk = np.arange(1, 40)
jj = np.arange(0, 898)
ii = np.arange(0, 398)
# box where changes took place (in all salish sea takes too much time, and not necessary I think)
pointsi = np.arange(50, 155).astype(int)
pointsj = np.arange(60, 100).astype(int)

#SSn = np.ones((1, 40, 898, 398))
for k in kk:
    for j in pointsj:
        for i in pointsi:
            diffsn = Sn[:, k, j, i] - Sn[:, k-1, j, i]
            if diffsn < 0:
                Sn[:, k, j, i] = Sn[:, k-1, j, i]
#            else:
#                sn[:, k, j, i] = sn[:, k, j, i]

Sn.shape


(1, 40, 898, 398)

In [45]:
for k in kk:
    for j in pointsj:
        for i in pointsi:
            diffsb = Sb[:, k, j, i] - Sb[:, k-1, j, i]
            if diffsb < 0:
                Sb[:, k, j, i] = Sb[:, k-1, j, i]

Sb.shape

(1, 40, 898, 398)

In [46]:
dfnew2 = dfnew
dfnew2['sn'] = Sn
dfnew2['sb'] = Sb


In [47]:
dfnew2.to_netcdf('/ocean/atall/MOAD/Model/runs/restart/31dec09/restart_PHYBIO_06324480_31dec09_bathy202605_snsbCorrected.nc', mode='w')

In [48]:
dfA = xr.open_dataset('/ocean/atall/MOAD/Model/runs/restart/31dec09/restart_PHYBIO_06324480_31dec09_bathy202605.nc')
dfB = xr.open_dataset('/ocean/atall/MOAD/Model/runs/restart/31dec09/restart_PHYBIO_06324480_31dec09_bathy202605_snsbCorrected.nc')
snA = dfA['sn']
snB = dfB['sn']